# 07 — DistilBERT V1

Entraînement de deux modèles légers à partir des pseudo-labels GPT-OSS 20B : détection multi-label des aspects et classification du sentiment.

Les métriques mesurent l’accord avec ces pseudo-labels, pas avec une vérité terrain humaine.

In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn pyarrow


In [ ]:
import time, random, numpy as np, pandas as pd, torch
from google.colab import files
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score, classification_report, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE=='cuda': print('GPU:', torch.cuda.get_device_name(0))


## 1. Chargement des Parquet

In [ ]:
uploaded = files.upload()
print('Fichiers reçus:', list(uploaded))


In [ ]:
parquet_files=[f for f in uploaded if f.lower().endswith('.parquet')]
assert len(parquet_files)>=2, 'Upload sample_2000.parquet et pairs.parquet.'
sample_path=pairs_path=None
for path in parquet_files:
    df_probe=pd.read_parquet(path)
    cols=set(df_probe.columns)
    if {'segment_id','segment_text','review_id'}.issubset(cols) and 'aspect_id' not in cols:
        sample_path=path
    if {'segment_id','aspect_id','sentiment'}.issubset(cols):
        pairs_path=path
print('Sample:', sample_path); print('Pairs:', pairs_path)
assert sample_path and pairs_path
sample=pd.read_parquet(sample_path); pairs=pd.read_parquet(pairs_path)
print('sample', sample.shape, '| pairs', pairs.shape)


In [ ]:
ASPECTS=['efficacy_results','hydration_dryness','texture_finish','irritation_sensitivity','acne_breakouts','fragrance_smell','application_absorption','packaging','price_value']
SENTIMENTS=['negative','neutral','positive']
assert sample['segment_id'].is_unique
assert set(pairs['aspect_id'].dropna().unique()).issubset(set(ASPECTS))
assert set(pairs['sentiment'].dropna().unique()).issubset(set(SENTIMENTS))
print('Segments:', len(sample)); print('Reviews:', sample['review_id'].nunique()); print('Pairs ABSA:', len(pairs))
display(pairs['aspect_id'].value_counts().to_frame('n'))


## 2. Split par `review_id`

In [ ]:
groups=sample['review_id'].astype(str)
gss=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED)
train_val_idx,test_idx=next(gss.split(sample,groups=groups))
train_val=sample.iloc[train_val_idx].copy(); test_sample=sample.iloc[test_idx].copy()
gss2=GroupShuffleSplit(n_splits=1,test_size=0.1765,random_state=SEED+1)
tr_rel,val_rel=next(gss2.split(train_val,groups=train_val['review_id'].astype(str)))
train_sample=train_val.iloc[tr_rel].copy(); val_sample=train_val.iloc[val_rel].copy()
print('Train',len(train_sample),'Val',len(val_sample),'Test',len(test_sample))
assert set(train_sample.review_id).isdisjoint(set(val_sample.review_id))
assert set(train_sample.review_id).isdisjoint(set(test_sample.review_id))
assert set(val_sample.review_id).isdisjoint(set(test_sample.review_id))


# A — Détection des aspects

In [ ]:
segment_aspects=(pairs.groupby('segment_id')['aspect_id'].apply(lambda s: sorted(set(s.astype(str)))).to_dict())
def make_aspect_df(df):
    out=df[['segment_id','review_id','segment_text']].copy()
    out['aspects']=out['segment_id'].map(segment_aspects).apply(lambda x: x if isinstance(x,list) else [])
    return out
train_aspect=make_aspect_df(train_sample); val_aspect=make_aspect_df(val_sample); test_aspect=make_aspect_df(test_sample)
mlb=MultiLabelBinarizer(classes=ASPECTS); mlb.fit([ASPECTS])
for d in [train_aspect,val_aspect,test_aspect]: d['labels']=list(mlb.transform(d['aspects']).astype(np.float32))
print(train_aspect.shape); display(train_aspect.head())


In [ ]:
MODEL_NAME='distilbert-base-uncased'; MAX_LENGTH=160
aspect_tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
aspect_model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=len(ASPECTS),problem_type='multi_label_classification',id2label={i:a for i,a in enumerate(ASPECTS)},label2id={a:i for i,a in enumerate(ASPECTS)})
def tok_aspect(batch): return aspect_tokenizer(batch['segment_text'],truncation=True,max_length=MAX_LENGTH)
def to_ds(df):
    ds=Dataset.from_pandas(df[['segment_text','labels']].reset_index(drop=True))
    return ds.map(tok_aspect,batched=True)
train_aspect_ds,to_val,to_test=to_ds(train_aspect),to_ds(val_aspect),to_ds(test_aspect)
aspect_collator=DataCollatorWithPadding(aspect_tokenizer)
TH=0.5
def sigmoid(x): return 1/(1+np.exp(-x))
def aspect_metrics(ep):
    logits,labels=ep; preds=(sigmoid(logits)>=TH).astype(int); labels=labels.astype(int)
    pm,rm,fm,_=precision_recall_fscore_support(labels,preds,average='micro',zero_division=0)
    pM,rM,fM,_=precision_recall_fscore_support(labels,preds,average='macro',zero_division=0)
    return {'precision_micro':pm,'recall_micro':rm,'f1_micro':fm,'precision_macro':pM,'recall_macro':rM,'f1_macro':fM,'exact_match':float(np.mean(np.all(preds==labels,axis=1)))}


In [ ]:
aspect_args=TrainingArguments(output_dir='/content/aspect_student',learning_rate=2e-5,per_device_train_batch_size=16,per_device_eval_batch_size=32,num_train_epochs=4,weight_decay=0.01,eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='f1_micro',greater_is_better=True,logging_steps=20,report_to='none',fp16=torch.cuda.is_available(),seed=SEED)
aspect_trainer=Trainer(model=aspect_model,args=aspect_args,train_dataset=train_aspect_ds,eval_dataset=to_val,tokenizer=aspect_tokenizer,data_collator=aspect_collator,compute_metrics=aspect_metrics)
aspect_trainer.train()


In [ ]:
print('=== ASPECT TEST ===')
metrics=aspect_trainer.evaluate(to_test)
for k,v in metrics.items(): print(k, round(v,4) if isinstance(v,float) else v)
pred=aspect_trainer.predict(to_test); pb=(sigmoid(pred.predictions)>=TH).astype(int); tb=pred.label_ids.astype(int)
rows=[]
for i,a in enumerate(ASPECTS):
    p,r,f,_=precision_recall_fscore_support(tb[:,i],pb[:,i],average='binary',zero_division=0)
    rows.append({'aspect_id':a,'precision':p,'recall':r,'f1':f,'support_positive':int(tb[:,i].sum())})
aspect_per_class=pd.DataFrame(rows).sort_values('f1',ascending=False); display(aspect_per_class)


# B — Sentiment conditionné par l’aspect

In [ ]:
pair_meta=pairs.merge(sample[['segment_id','review_id','segment_text']].drop_duplicates('segment_id'),on='segment_id',how='left',validate='many_to_one')
train_reviews=set(train_sample.review_id.astype(str)); val_reviews=set(val_sample.review_id.astype(str)); test_reviews=set(test_sample.review_id.astype(str))
pair_meta['_r']=pair_meta.review_id.astype(str)
train_sent=pair_meta[pair_meta._r.isin(train_reviews)].copy(); val_sent=pair_meta[pair_meta._r.isin(val_reviews)].copy(); test_sent=pair_meta[pair_meta._r.isin(test_reviews)].copy()
sent2id={'negative':0,'neutral':1,'positive':2}; id2sent={v:k for k,v in sent2id.items()}
def prep(df):
    out=df.copy(); out['model_text']='[ASPECT] '+out.aspect_id.astype(str)+' [TEXT] '+out.segment_text.astype(str); out['labels']=out.sentiment.map(sent2id); return out
train_sent,val_sent,test_sent=prep(train_sent),prep(val_sent),prep(test_sent)
print('Pairs sentiment:',len(train_sent),len(val_sent),len(test_sent))


In [ ]:
sent_tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
sent_model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=3,id2label=id2sent,label2id=sent2id)
def tok_sent(batch): return sent_tokenizer(batch['model_text'],truncation=True,max_length=MAX_LENGTH)
def to_sent_ds(df):
    ds=Dataset.from_pandas(df[['model_text','labels']].reset_index(drop=True)); return ds.map(tok_sent,batched=True)
tr_s,va_s,te_s=to_sent_ds(train_sent),to_sent_ds(val_sent),to_sent_ds(test_sent)
sent_collator=DataCollatorWithPadding(sent_tokenizer)
def sent_metrics(ep):
    logits,labels=ep; preds=np.argmax(logits,axis=-1)
    return {'accuracy':accuracy_score(labels,preds),'f1_macro':f1_score(labels,preds,average='macro',zero_division=0),'f1_weighted':f1_score(labels,preds,average='weighted',zero_division=0)}


In [ ]:
sent_args=TrainingArguments(output_dir='/content/sentiment_student',learning_rate=2e-5,per_device_train_batch_size=16,per_device_eval_batch_size=32,num_train_epochs=4,weight_decay=0.01,eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='f1_macro',greater_is_better=True,logging_steps=20,report_to='none',fp16=torch.cuda.is_available(),seed=SEED)
sent_trainer=Trainer(model=sent_model,args=sent_args,train_dataset=tr_s,eval_dataset=va_s,tokenizer=sent_tokenizer,data_collator=sent_collator,compute_metrics=sent_metrics)
sent_trainer.train()


In [ ]:
print('=== SENTIMENT TEST ===')
sm=sent_trainer.evaluate(te_s)
for k,v in sm.items(): print(k, round(v,4) if isinstance(v,float) else v)
sp=sent_trainer.predict(te_s); y_pred=np.argmax(sp.predictions,axis=-1); y_true=sp.label_ids
print(classification_report(y_true,y_pred,target_names=['negative','neutral','positive'],zero_division=0))
cm=confusion_matrix(y_true,y_pred,labels=[0,1,2]); cm_df=pd.DataFrame(cm,index=['gold_negative','gold_neutral','gold_positive'],columns=['pred_negative','pred_neutral','pred_positive']); display(cm_df)


## 3. Vitesse d’inférence

In [ ]:
model=aspect_trainer.model.to(DEVICE); model.eval(); texts=test_aspect.segment_text.tolist()[:min(200,len(test_aspect))]
enc=aspect_tokenizer(texts,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt'); enc={k:v.to(DEVICE) for k,v in enc.items()}
if DEVICE=='cuda': torch.cuda.synchronize()
t0=time.perf_counter()
with torch.no_grad(): _=model(**enc)
if DEVICE=='cuda': torch.cuda.synchronize()
elapsed=time.perf_counter()-t0
print('Segments:',len(texts)); print('Temps total:',round(elapsed,4),'s'); print('ms/segment:',round(1000*elapsed/len(texts),2)); print('segments/s:',round(len(texts)/elapsed,1))


## 4. Sauvegarde

In [ ]:
ASPECT_DIR='/content/student_aspect_distilbert'; SENT_DIR='/content/student_sentiment_distilbert'
aspect_trainer.save_model(ASPECT_DIR); aspect_tokenizer.save_pretrained(ASPECT_DIR)
sent_trainer.save_model(SENT_DIR); sent_tokenizer.save_pretrained(SENT_DIR)
aspect_per_class.to_csv('/content/aspect_metrics_per_class.csv',index=False); cm_df.to_csv('/content/sentiment_confusion_matrix.csv')
!zip -qr /content/sephora_absa_student_models.zip /content/student_aspect_distilbert /content/student_sentiment_distilbert /content/aspect_metrics_per_class.csv /content/sentiment_confusion_matrix.csv
files.download('/content/sephora_absa_student_models.zip')


## Résultat

Cette V1 sert de baseline avant correction du déséquilibre de classes et ajustement des seuils.